In [ ]:
""" in this i wanted to develop a Enterprise Knowledge Assistant which takes multiple document with different formats.
what to be developed:
1 a system with multiple document loaders
2 a different vector database for storing the embeddings
3 
"""

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
import os
import langchain
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
load_dotenv(override=True)
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
doc_path=os.getenv("doc_path")
print(doc_path)



In [ ]:

print(langchain.__version__)

In [ ]:
!pip install docx2txt

In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    UnstructuredExcelLoader,
    UnstructuredPowerPointLoader
)


In [ ]:
!pip install "unstructured[xlsx]"
!pip install "unstructured[pptx]"

In [ ]:
def load_documents(docspath):
    docs=[]

    for file in os.listdir(docspath):
        path=os.path.join(docspath, file)
        if file.endswith(".txt"):
            loader = TextLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".pdf"):
            loader = PyPDFLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".docx"):
            loader = Docx2txtLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".xlsx"):
            loader = UnstructuredExcelLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".pptx"):
            loader = UnstructuredPowerPointLoader(path)
            docs.extend(loader.load())
        else:
            continue
    
    return docs


In [ ]:
docsx=load_documents(doc_path)

In [ ]:
type(docsx)

In [ ]:
print(type(docsx))
print(len(docsx))
print(type(docsx[0]))

In [ ]:
docsx[:4]

In [ ]:
len(docsx)

In [ ]:
for d in docsx:
    print(d.metadata.get("source"))
    #print(d.page_content)
    print("---------------------------------------------------")

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits=splitter.split_documents(docsx)

In [ ]:
len(splits)

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

In [ ]:
!pip install langchain-qdrant

In [ ]:
from langchain_qdrant import QdrantVectorStore

qdrant = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
     location=":memory:",
    collection_name="my_collection",
)

print("Uploaded successfully")

In [ ]:
!pip install rank_bm25 langchain-classic

In [ ]:
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bmretreiver=BM25Retriever.from_documents(splits)
bmretreiver.k=3



In [ ]:
dense_retriever = qdrant.as_retriever(search_kwargs={"k": 3})

finalretriever = EnsembleRetriever(
    retrievers=[bmretreiver, dense_retriever],
    weights=[0.6, 0.4]
)

In [ ]:
result = finalretriever.invoke("What should I do if my laptop is stolen?")
result

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def reranking(question,result,top_k=3):
    # Extract the text from the result
    texts = [doc.page_content for doc in result]
    
    # Create pairs of (question, text) for scoring
    pairs = [(question, text) for text in texts]
    
    # Get scores from the reranker
    scores = reranker.predict(pairs)
    
    # Combine scores with the original documents
    scored_results = list(zip(result, scores))
    
    # Sort by score in descending order
    scored_results.sort(key=lambda x: x[1], reverse=True)
    
    # Return the top_k results
    return [doc for doc, score in scored_results[:top_k]]


In [ ]:
check=reranking("What should I do if my laptop is stolen?", result, top_k=3)
check
for i, doc in enumerate(check):
    print(f"--- Result {i+1} ---")
    #print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:200])
    print()

In [ ]:
openai=ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions based on the provided context.while answering provide the metadata as well"),
    ("human", "{metadata}\n\n{context}\n\nQuestion: {question}\nAnswer:")
])

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])
def ch_metadata(docs):
    return "\n\n".join([f"Source: {doc.metadata.get('source')}" for doc in docs])

from langchain_core.runnables import RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": finalretriever | format_docs, "metadata": finalretriever | ch_metadata, "question": RunnablePassthrough()}
    | prompt
    | openai
    | StrOutputParser()
)

answer = rag_chain.invoke("what should I do if my laptop is stolen?")
print(answer)

print("the below answer is generated by the RAG pipeline using the ensemble retriever and reranking model")

def retrieve_rank(question):
    result = finalretriever.invoke(question)
    return reranking(question, result, top_k=3)

rag_chains = (
    {"context": RunnableLambda(retrieve_rank) | format_docs, "metadata": RunnableLambda(retrieve_rank) | ch_metadata, "question": RunnablePassthrough()}
    | prompt
    | openai
    | StrOutputParser()
)

answers = rag_chains.invoke("what should I do if my laptop is stolen?")
print(answers)

In [ ]:
before_docs = finalretriever.invoke("what should I do if my laptop is stolen?")
after_docs = reranking("what should I do if my laptop is stolen?", before_docs, top_k=3)

print("Before ranking")
for d in before_docs:
    print(d.metadata.get("source", "").split("/")[-1])

print("\nafter ranking")
for d in after_docs:
    print(d.metadata.get("source", "").split("/")[-1])

In [ ]:
for d in docsx:
    print(d.metadata.get("source"))

In [ ]:
questions = [
    "What should I do if my laptop is stolen?",
    "How do I reset or rotate my password?",
    "How do I install the VPN client?",
    "How many PTO days do I get per year?",
    "What's the hotel reimbursement limit for business travel?",
    "How do I roll back a bad production deployment?",
    "What are the standard working hours?",
    "What authentication method do TechNova APIs use?",
]

for q in questions:
    z = finalretriever.invoke(q)
    sources = [doc.metadata.get("source") for doc in z]
    print(q)
    print(sources)
    print()

In [ ]:
for q in questions:
    z = finalretriever.invoke(q)
    reranked = reranking(q, z, top_k=3)
    sources = [doc.metadata.get("source") for doc in reranked]
    print(q)
    print(sources)
    print()